In [ ]:
# -------- Import the Libraries --------

In [1]:
from typing import List
import json
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import MessageGraph, END
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage


# -------- LLM --------

In [3]:
llm = ChatOllama(
    model="gemma4:e2b",
    temperature=0,
)
llm_qwen = ChatOllama(model="qwen3:4b", temperature=0)


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "Gemeni-API"  # Replace with your actual API key

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=API_KEY
)

C:\Users\aa\anaconda3\envs\gpu_env\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


# -------- Prompt Template --------

In [6]:
prompt_template = ChatPromptTemplate.from_messages([
(
"system",
"""
You are a neutral, evidence-informed nutrition assistant.
Your job is to help the user design a balanced meal using ONLY the ingredients they already have at home.

Rules:
- No dietary ideology (no carnivore, no vegan, no keto bias).
- Use general nutrition principles: balance protein, fiber, healthy fats, and reasonable carbs.
- If the user mentions health goals (e.g., blood sugar, heart health), gently favor lower added sugars, more fiber, and healthier fats.
- Do NOT give medical diagnoses or claim to treat disease.

CRITICAL VALIDATION RULE:
You MUST fill every field in the AnswerQuestion tool. 
If ANY field is empty, your entire output is INVALID and MUST be regenerated.

You MUST fill:
- answer
- reflection.missing
- reflection.superfluous
- search_queries (1–3 items)

Reflection Requirements:
- The 'reflection' field MUST NOT be empty.
- You must analyze your own answer, identifying:
  - missing information
  - unnecessary/superfluous information

Search Queries:
After the reflection, list 1–3 search queries separately for researching:
  - nutritional value of suggested ingredients
  - impact on satiety, blood sugar, or general health

Your response must follow these steps:
1. {first_instruction}
2. Design a concrete meal (or 1–2 options) using ONLY the ingredients the user has.
3. Briefly explain why this meal is reasonably balanced (protein, fiber, fats, carbs).
4. Reflect and critique your answer. Be honest about limitations (e.g., missing vegetables, too little fiber, etc.).
5. After the reflection, list 1–3 search queries (not inside the reflection).
"""
)
,
    MessagesPlaceholder(variable_name="messages"),
    (
        "system", 
        "Answer the user's question above using the required format, focusing on practical, balanced meals from available ingredients."
    ),
])


# -------- First responder (Draft) --------

In [8]:
first_responder_prompt = prompt_template.partial(
    first_instruction="Provide a detailed ~250 word answer designing a balanced meal using the user's available ingredients."
)

In [9]:
temp_chain = first_responder_prompt | llm

In [10]:
question = """
I have these ingredients at home: eggs, oats, yogurt, tuna, tomatoes, olive oil, whole-grain bread, cheese.
I want a breakfast that is reasonably good for blood sugar and heart health. Design a meal using only these.
"""

response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print("---- Raw Draft (no tools) ----")
print(response.content)

---- Raw Draft (no tools) ----
Here is a balanced breakfast meal design using only your available ingredients, keeping blood sugar and heart health in mind:

**Savory Egg & Tuna Toast with a Yogurt & Oat Side**

This meal combines protein, fiber, and healthy fats to help promote satiety and provide sustained energy, which is beneficial for managing blood sugar levels and supporting heart health.

1.  **The Savory Toast:**
    *   **Ingredients:** 1-2 eggs, a small portion of tuna (drained if oil-packed, or from water), 1 slice of whole-grain bread, a few slices or chopped pieces of tomato, a drizzle of olive oil, and a thin slice of cheese (optional).
    *   **Preparation:** Heat a small pan with a drizzle of olive oil. Scramble the eggs, incorporating the drained tuna and finely chopped tomatoes during cooking. Toast your whole-grain bread. If using cheese, you can melt a thin slice onto the hot toast. Top the toast with your egg, tuna, and tomato scramble.
    *   **Why it works:** 

# -------- Tool schema --------

In [ ]:
class Reflection(BaseModel):
"""Encapsulates the self-critique mechanism to identify missing or irrelevant information in the draft."""
    missing: str = Field(description="What information is missing")
    superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
"""Represents the final structured output, containing the primary response, self-critique, and supporting search queries."""
    answer: str = Field(description="IMPORTANT: Main response to the question (meal design + explanation)")
    reflection: Reflection = Field(description="IMPORTANT: Self-critique of the answer")
    search_queries: List[str] = Field(description="IMPORTANT: Queries for additional research")

# -------- Initial chain with tool binding --------

In [ ]:
# Bind the structured output schema (AnswerQuestion) to the LLM 
# to enforce a strict JSON-like output format for every response.
initial_chain = first_responder_prompt | llm.bind_tools(tools=[AnswerQuestion])

# Invoke the chain with the user's question, triggering the agent's 
# first attempt to generate a draft and populate the structured tool fields.
response = initial_chain.invoke({"messages": [HumanMessage(content=question)]})

print("--- Full Structured Output ---")
# Print the extracted tool calls to verify that the LLM successfully 
# followed the schema and generated the required fields (answer, reflection, search_queries).
print(response.tool_calls)

--- Full Structured Output ---
[{'name': 'AnswerQuestion', 'args': {'answer': 'Here is a balanced breakfast designed using only your available ingredients, keeping blood sugar and heart health in mind:\n\n**Meal Suggestion: Savory Egg & Tuna Toast with Tomato and a touch of Cheese**\n\nPrepare two slices of **whole-grain bread** as toast. While toasting, scramble two **eggs** with a tiny drizzle of **olive oil**. Flake about half a can of **tuna** (drained if packed in water, or use the oil if packed in olive oil and you prefer that flavor) and mix it gently with a few slices of chopped **tomato**. Once the toast is ready, spread the scrambled eggs on top, then layer the tuna and tomato mixture. Finish with a very thin slice or a small sprinkle of **cheese** and another light drizzle of **olive oil** over the tomatoes.\n\n**Why this meal is reasonably balanced for blood sugar and heart health:**\n\n*   **Protein Power (Eggs, Tuna, Cheese):** This meal is rich in protein from eggs, tuna

In [15]:
if not response.tool_calls:
    print("Model did not return a tool call. Got content instead:")
    print(response.content)
    raise SystemExit()

In [16]:
answer_content = response.tool_calls[0]['args']['answer']
print("--- Initial Answer ---")
print(answer_content)

--- Initial Answer ---
Here is a balanced breakfast designed using only your available ingredients, keeping blood sugar and heart health in mind:

**Meal Suggestion: Savory Egg & Tuna Toast with Tomato and a touch of Cheese**

Prepare two slices of **whole-grain bread** as toast. While toasting, scramble two **eggs** with a tiny drizzle of **olive oil**. Flake about half a can of **tuna** (drained if packed in water, or use the oil if packed in olive oil and you prefer that flavor) and mix it gently with a few slices of chopped **tomato**. Once the toast is ready, spread the scrambled eggs on top, then layer the tuna and tomato mixture. Finish with a very thin slice or a small sprinkle of **cheese** and another light drizzle of **olive oil** over the tomatoes.

**Why this meal is reasonably balanced for blood sugar and heart health:**

*   **Protein Power (Eggs, Tuna, Cheese):** This meal is rich in protein from eggs, tuna, and cheese. Protein is crucial for satiety, helping you feel f

In [17]:
reflection_content = response.tool_calls[0]['args']['reflection']
print("--- Reflection Answer ---")
print(reflection_content)

--- Reflection Answer ---
{'superfluous': 'None. All information provided was directly relevant to designing the meal and explaining its nutritional benefits.', 'missing': 'The meal suggestion did not directly incorporate oats or yogurt. While the provided meal is balanced, suggesting that a small serving of plain yogurt could be a beneficial addition for probiotics and extra protein, or that oats could form a separate, alternative breakfast option, would have made the answer more comprehensive in utilizing all ingredients.'}


In [18]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("--- Search Queries ---")
print(search_queries)

--- Search Queries ---
['Nutritional benefits of whole-grain bread for blood sugar', 'Omega-3 fatty acids in tuna and heart health', 'Fiber content in tomatoes and eggs']


# -------- Build state list --------

In [ ]:
# Initialize a conversation history list to track the message flow
response_list: List[BaseMessage] = []
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [ ]:
# Extract the tool call generated by the LLM
tool_call = response.tool_calls[0]
# Retrieve the list of search queries proposed by the model for external grounding
search_queries = tool_call["args"].get("search_queries", [])
print("Search queries from tool_call:", search_queries)

Search queries from tool_call: ['Nutritional benefits of whole-grain bread for blood sugar', 'Omega-3 fatty acids in tuna and heart health', 'Fiber content in tomatoes and eggs']


# -------- Tavily tool --------

In [ ]:
# Initialize the Tavily search tool to fetch external data. 
# This tool enables the agent to perform real-time research, 
# ensuring all responses are grounded in accurate and up-to-date information.
tavily_tool = TavilySearchResults(
    max_results=3,
    tavily_api_key="Tavily-API"  # Replace with your actual API key
)

C:\Users\aa\AppData\Local\Temp\ipykernel_16188\1286979189.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(


# -------- execute_tools --------

In [ ]:
def execute_tools(state: List[BaseMessage]) -> List[BaseMessage]:
    """
    Executes search queries proposed by the LLM and formats the results 
    as ToolMessages to be processed in the next step of the LangGraph.
    """
    last_ai_message = state[-1]
    tool_messages: List[ToolMessage] = []
    
    # Check if the last message contains tool calls generated by the agent
    if not getattr(last_ai_message, "tool_calls", None):
        return tool_messages

    # Process each tool call and fetch results using the Tavily search tool
    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            
            # Execute each search query and aggregate the retrieved information
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            
            # Encapsulate the gathered research results into a ToolMessage
            tool_messages.append(
                ToolMessage(
                    content=json.dumps(query_results),
                    tool_call_id=call_id
                )
            )
    return tool_messages

# Invoke the tool execution logic and append the search results to the conversation history
tool_response = execute_tools(response_list)
response_list.extend(tool_response)

In [26]:
tool_response

[ToolMessage(content='{"Nutritional benefits of whole-grain bread for blood sugar": [{"title": "Best Bread for Diabetes", "url": "https://sesamecare.com/blog/best-bread-for-diabetics", "content": "To make a healthy bread choice, individuals, especially those managing conditions like diabetes, should look for bread labeled \\"100% whole wheat\\" or \\"100% whole grain\\" to ensure they are getting the full nutritional benefits of whole grains.\\n\\nUnlike refined white bread, which can cause rapid spikes in blood sugar levels, whole grain, and whole wheat bread contain complex carbohydrates and fiber. These components slow down glucose absorption, leading to a more gradual and steady rise in blood sugar, making them a better choice for individuals with diabetes. Additionally, these types of bread are high-fiber, which promotes satiety and helps with weight management, a critical factor in diabetes management.\\n\\n### Fiber [...] ### Whole grain and whole wheat\\n\\nThe American Diabete

# -------- Revisor instructions --------

In [28]:
revise_instructions = """
Revise your previous answer using the new information, applying the rigor and evidence-based approach of Dr. David Attia.
- Incorporate the previous critique to add clinically relevant information, focusing on mechanistic understanding and individual variability.
- You MUST include numerical citations referencing peer-reviewed research, randomized controlled trials, or meta-analyses to ensure medical accuracy.
- Distinguish between correlation and causation, and acknowledge limitations in current research.
- Address potential biomarker considerations (lipid panels, inflammatory markers, and so on) when relevant.
- Add a "References" section to the bottom of your answer Use ONLY the URLs found in the provided 'ToolMessage' context.
- Use the previous critique to remove speculation and ensure claims are supported by high-quality evidence.
- Keep response under 250 words with precision over volume.
- When discussing nutritional interventions, consider metabolic flexibility, insulin sensitivity, and individual response variability.
"""


In [29]:
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)
revisor_chain = revisor_prompt | llm.bind_tools(tools=[AnswerQuestion])


In [ ]:
import time

def revisor_with_delay(state):
    """
    Invokes the revision chain to refine the agent's output. 
    Includes a 2-minute delay to comply with API rate limits or to 
    allow external processes to stabilize before proceeding.
    """
    print("Omar") # Debugging log to track the agent's progress
    time.sleep(120)
    
    return revisor_chain.invoke({"messages": state})

In [31]:
response = revisor_chain.invoke({"messages": response_list})

In [32]:
#response = revisor_chain.invoke({"messages": response_list})
print("--- Revised Answer (Structured) ---")
if response.tool_calls:
    print(response.tool_calls[0]['args'])
else:
    print("No tool call in revisor response. Got content:")
    print(response.content)

--- Revised Answer (Structured) ---
{'answer': 'Here is a balanced breakfast using only your available ingredients, designed with blood sugar and heart health in mind:\n\n**Meal Suggestion: Savory Egg & Tuna Toast with Tomato and Yogurt**\n\nPrepare two slices of **whole-grain bread** as toast. While toasting, scramble two **eggs** with a tiny drizzle of **olive oil**. Flake about half a can of **tuna** (drained) and mix it gently with a few slices of chopped **tomato**. Once the toast is ready, spread the scrambled eggs on top, then layer the tuna and tomato mixture. Finish with a very thin slice or a small sprinkle of **cheese** and another light drizzle of **olive oil** over the tomatoes. Serve with a small side of plain **yogurt**.\n\n**Why this meal is reasonably balanced for blood sugar and heart health:**\n\nThis meal emphasizes a synergistic approach to metabolic health. The **whole-grain bread** provides complex carbohydrates and dietary fiber, which are digested slowly, leadi

In [33]:
response_list.append(response)

# -------- Graph + loop --------

In [ ]:
# Define the maximum number of reflexions to prevent infinite loops.
MAX_ITERATIONS = 2

# Global counter to track the current iteration of the Reflexion Loop.
iteration_counter = 0

def event_loop(state):
    """
    Acts as a conditional edge in the LangGraph.
    Decides whether to continue the research/revision process or terminate 
    the flow based on the iteration limit.
    """
    global iteration_counter

    iteration_counter += 1

    print("Iterations =", iteration_counter)

    # Terminate the loop if the maximum number of iterations has been reached
    if iteration_counter >= MAX_ITERATIONS:
        return END

    # Otherwise, route to the tool execution/revision phase
    return "execute_tools"

In [ ]:
# Initialize the MessageGraph to define the agent's workflow
graph = MessageGraph()

# Add nodes to the graph representing different stages of the agent's logic
graph.add_node("respond", initial_chain)        # Node for initial response generation
graph.add_node("execute_tools", execute_tools)  # Node for fetching real-time data
graph.add_node("revisor", revisor_with_delay)   # Node for critique and revision

# Define the sequential flow between nodes
graph.add_edge("respond", "execute_tools")      # Move from response to data execution
graph.add_edge("execute_tools", "revisor")      # Move from data execution to revision

# Define conditional edges to control the iteration loop
# The 'revisor' node decides whether to continue iterating or terminate based on the event_loop logic
graph.add_conditional_edges(
    "revisor",
    event_loop,
    {
        "execute_tools": "execute_tools", # Continue refining
        END: END,                        # Finish the process
    }
)

# Set the entry point of the graph
graph.set_entry_point("respond")

C:\Users\aa\AppData\Local\Temp\ipykernel_16188\1484313378.py:1: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = MessageGraph()


In [ ]:
# Compile the graph into an executable application
app = graph.compile()

# Invoke the agent with a specific query, initiating the defined workflow 
# (Respondent -> Tools -> Revisor -> Conditional Loop -> END)
responses = app.invoke(
    """
    I have these ingredients at home: eggs, oats, yogurt, tuna, tomatoes, olive oil, whole-grain bread, cheese.
    I want a breakfast that is reasonably good for blood sugar and heart health. Design a meal using only these.
    """
)

Omar
Iterations = 1
Omar
Iterations = 2


In [ ]:
# Print the initial draft generated by the first node in the graph
print("--- Initial Draft Answer (from graph) ---")
initial_answer = responses[1].tool_calls[0]['args']['answer']
print(initial_answer)
print("\n")

# Extract and display all intermediate and final revised answers 
# from the conversation history, traversing the messages in reverse order.
print("--- Intermediate and Final Revised Answers ---")
answers = []

for msg in reversed(responses):
    # Check if the message contains tool calls (where our answers are stored)
    if getattr(msg, 'tool_calls', None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get('args', {}).get('answer')
            if answer:
                answers.append(answer)

# Print the collected answers with labels to show the evolution of the output
for i, ans in enumerate(answers):
    # Label the first extracted item as 'Final' and subsequent ones as 'Intermediate'
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{ans}\n")

--- Initial Draft Answer (from graph) ---
Here is a balanced breakfast designed using only your available ingredients, focusing on blood sugar and heart health:

**Savory Oatmeal with Poached Egg, Diced Tomatoes, and Olive Oil**

**Preparation:**
1.  Cook a serving of oats with water according to package directions.
2.  While the oats are cooking, poach or boil an egg to your desired doneness.
3.  Dice a fresh tomato.

**Assembly:**
Pour the cooked oats into a bowl. Place the poached egg on top. Scatter the diced tomatoes over the oats and egg. Finish with a small drizzle of olive oil. If you'd like, a very small sprinkle of cheese can be added for extra flavor, but keep the amount minimal to manage saturated fat intake.

**Why this meal is balanced for blood sugar and heart health:**
*   **Complex Carbohydrates and Fiber (Oats, Tomatoes):** Oats are an excellent source of soluble fiber, which is crucial for managing blood sugar by slowing down glucose absorption and can help lower LDL